In [ ]:
# import system libs
import os
import time
import shutil
import pathlib
import itertools
import random
import warnings

# import data handling tools
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras.applications import Xception
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras import regularizers

import sys
import os

# Add the parent directory to sys.path so Python can find 'pipeline_helpers'
sys.path.append(os.path.abspath('..'))

from pipeline_helpers.data_preprocessing import 


# set styles and filter warnings
sns.set_style('darkgrid')
warnings.filterwarnings("ignore")


ImportError: cannot import name 'your_function_name' from 'pipeline_helpers.data_preprocessing' (c:\Users\User\Desktop\Kaggle\MRI_brain_tumor_prediction\MRI_CNN_Architectures\pipeline_helpers\data_preprocessing.py)

In [2]:

import os

# Base path relative to notebook location
base_dir = os.path.abspath(os.path.join("..", "archive"))

In [3]:
import sys
import os

# Add the parent directory of `pipeline_helpers` to the Python path
sys.path.append(os.path.abspath(".."))
from pipeline_helpers.data_preprocessing import prepare_generators


train_gen, val_gen, test_gen, class_dict = prepare_generators(base_dir)


Found 4569 validated image filenames belonging to 4 classes.
Found 1143 validated image filenames belonging to 4 classes.
Found 1311 validated image filenames belonging to 4 classes.


In [12]:
def build_model(input_shape=(224, 224, 3), num_classes=4):
    base_model = MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape, pooling='max')
    base_model.trainable = False

    model = Sequential([
        base_model,
        Flatten(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.25),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer=Adamax(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy', Precision(), Recall()])
    return model


In [13]:
print("\n Training model on train/validation split")



# Build the model
model = build_model()

# Train the model
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    verbose=1
)



 Training model on train/validation split
Epoch 1/10
143/143 [==============================] - 212s 1s/step - loss: 1.4981 - accuracy: 0.5964 - precision_2: 0.6279 - recall_2: 0.5559 - val_loss: 0.6693 - val_accuracy: 0.7463 - val_precision_2: 0.8424 - val_recall_2: 0.6360
Epoch 2/10
143/143 [==============================] - 175s 1s/step - loss: 0.7402 - accuracy: 0.7120 - precision_2: 0.7612 - recall_2: 0.6461 - val_loss: 0.5652 - val_accuracy: 0.7988 - val_precision_2: 0.8702 - val_recall_2: 0.7270
Epoch 3/10
143/143 [==============================] - 174s 1s/step - loss: 0.6484 - accuracy: 0.7514 - precision_2: 0.7927 - recall_2: 0.7006 - val_loss: 0.5411 - val_accuracy: 0.8066 - val_precision_2: 0.8750 - val_recall_2: 0.7288
Epoch 4/10
143/143 [==============================] - 172s 1s/step - loss: 0.5755 - accuracy: 0.7722 - precision_2: 0.8065 - recall_2: 0.7306 - val_loss: 0.4448 - val_accuracy: 0.8268 - val_precision_2: 0.8720 - val_recall_2: 0.7865
Epoch 5/10
143/143 [=====

In [21]:
import sys
import os

# Add the root directory to sys.path
sys.path.append(os.path.abspath('..'))

from pipeline_helpers.evaluation import (
    save_model_architecture,
    save_model_weights,
    plot_training_metrics,
    evaluate_and_save_results,
    save_confusion_matrix,
    predict_and_save_plot
)


In [22]:
output_dir = '../outputs/models/light_model_evaluation'
os.makedirs(output_dir, exist_ok=True)


In [24]:
# Save model architecture diagram
save_model_architecture(model, output_dir, 'architecture.png')

# Save model weights
save_model_weights(model, output_dir, 'light_model')

# Plot training metrics
plot_training_metrics(history, os.path.join(output_dir, 'training_metrics.png'))

# Evaluate on all three sets
evaluate_and_save_results(model, train_gen, val_gen, test_gen, output_dir)

# Confusion matrix (for test set)
class_labels = list(train_gen.class_indices.keys())  # Get class names
save_confusion_matrix(model, test_gen, class_labels, os.path.join(output_dir, 'confusion_matrix.png'))


You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.
Model architecture image saved to: ../outputs/models/light_model_evaluation\architecture.png
Model saved as .h5 and .keras in: ../outputs/models/light_model_evaluation
Training metrics plot saved to: ../outputs/models/light_model_evaluation\training_metrics.png
41/41 [==============================] - 37s 898ms/step - loss: 0.4821 - accuracy: 0.8169 - precision_2: 0.8344 - recall_2: 0.7956
Train Set
Loss      : 0.2864
Accuracy  : 89.58%
Precision : 91.20%
Recall    : 88.01%
----------------------------------------
Validation Set
Loss      : 0.3216
Accuracy  : 88.54%
Precision : 90.03%
Recall    : 86.88%
----------------------------------------
Test Set
Loss      : 0.4821
Accuracy  : 81.69%
Precision : 83.44%
Recall    : 79.56%
----------------------------------------
41/41 [==============================] - 32s 767ms/step
Confusion matrix 

'              precision    recall  f1-score   support\n\n      glioma       0.97      0.65      0.78       300\n  meningioma       0.68      0.60      0.64       306\n     notumor       0.83      0.98      0.90       405\n   pituitary       0.81      0.98      0.89       300\n\n    accuracy                           0.82      1311\n   macro avg       0.82      0.80      0.80      1311\nweighted avg       0.82      0.82      0.81      1311\n'